# 1. Importacion de Librerias 
---
1. Traigo librerias para empezar, datos y scraping para conectar.
2. Sesion de requests lista quedo, y un plan de reintentos se definio
3. Si falla el Envi­o por un error, tres veces insiste con gran valor.
4. Se monta el adaptador con atencion, aplicando la regla a la conexion.
5. Luego de que revisa, un mensaje avisa que todo cargo con prisa.
***

In [9]:
# 1.1 Importacion de librerias necesarias
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import json
import re
from urllib.parse import urljoin, urlparse, quote
from datetime import datetime
import pandas as pd
import os
from typing import Dict,List,Optional,Tuple,Any
import hashlib

#1.2 para manejar errores en red 
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#1.3 Configuracion de sesion con reintentos
sesion = requests.Session()
sesion.headers.update({
    'User-Agent': 'web-scraping-challenge/1.0 (educational project)'
})
retry=Retry (total=3,
            backoff_factor=0.5,
            status_forcelist=[429,500,502,503,504])

#1.4 adaptador
adaptador=HTTPAdapter(max_retries=retry)
sesion.mount('http://', adaptador)
sesion.mount('https://', adaptador)

#1.5 crear carpetas datos si no existe
try:
    os.makedirs('datos', exist_ok=True)
    print("Directorio 'datos' Verificado/creado Correctamente.")
except PermissionError:
    print("Error: No tienes permisos para crear el directorio 'datos'.")
except FileNotFoundError:
    print("Error: la ruta especificada no es válida o alguna carpeta padre no existe")
except OSError as e:
    print(f"Error al crear el directorio {e}")

#1.6 aviso de librerias importadas correctamente 
print("Configuracion completada. librerias importadas")

Directorio 'datos' Verificado/creado Correctamente.
Configuracion completada. librerias importadas


# 2. Funciones auxiliares (HTML, espera, cache)
- Toma la URL y el HTML a explorar, y un BeautifulSoup te da para analizar.
- Pausa la ejecucion unos segundos al andar, para respetar li­mites y no sobrecargar.
- En memoria un diccionario va a crear, y repeticiones a la API logra evitar.

In [10]:
#2.1 Obtiene el contenido html de una pagina web, una url y lo devuelve, un objeto beutifulsoup.
def obtener_html(url:str, timeout:int=10)->Optional[BeautifulSoup]:
    try:
        respuesta = sesion.get(url, timeout=timeout)
        respuesta.raise_for_status()
        soup=BeautifulSoup(respuesta.content,'html.parser')
        print(f"Contenido HTML obtenido correctamente de {url}")
        return soup
    except requests.exceptions.HTTPError as e:
        if e.response.status_code==404:
            print(f" Pagina web no encontrada (404): {url}")
        else:
            print(f"Error HTTP Al obtener el contenido HTML {e.response.status_code} de: {url}")
    except requests.exceptions.Timeout:
        print(f"Tiempo de espera agotado al obtener el contenido HTML de: {url}")
    except requests.exceptions.RequestException as e:
        print(f" Error de red al obtener el contenido HTML de: {url}. Detalles: {e}")
    return None

#2.2 funcion para esperar un tiempo pausa para respetar limites de velocidad
def esperar(segundos: float =1):
    print(f" Esperando {segundos} segundos...")
    time.sleep(segundos)

def obtener_conexion():
    conexion=sqlite3.connect('libreria.db')
    conexion.execute ("PRAGMA foreign_keys = ON")
    return conexion

#2.4 cache de autores en (memoria)
cache_autores={}
MARCADOR_NO_ENCONTRADO="NO_ENCONTRADO"

def cargar_cache_autores():
    global cache_autores
    archivo_cache='datos/cache_autores.csv'

    if os.path.exists(archivo_cache):
        df = pd.read_csv(archivo_cache)

        for _, row in df.iterrows():
            nombre=row['nombre']

            if row.get('api_source')== MARCADOR_NO_ENCONTRADO:
                cache_autores[nombre]=None
                continue

            datos={
                'pais': row.get('pais'),
                'api_id': row.get('api_id'),
                'api_source': row.get('api_source')
            }
            for clave, valor in datos.items():
                if pd.isna(valor):
                    datos[clave] = None
            cache_autores[nombre]=datos

        print(f" Cache de autores cargada desde {archivo_cache}. Total autores: ({len(cache_autores)} en registros)")

#2.5 funcion para guardar cache de autores en un archivo csv
def guardar_cache_autores():
    registros=[]
    for nombre, datos in cache_autores.items():
        if datos is None:
            registros.append({
                'nombre': nombre,
                'pais': None,
                'api_id':None,
                'api_source':MARCADOR_NO_ENCONTRADO
            })
            continue

        registros.append({
            'nombre': nombre,
            'pais': datos.get('pais'),
            'api_id':datos.get('api_id'),
            'api_source':datos.get('api_source')
        })

    df=pd.DataFrame(registros, columns=['nombre','pais','api_id','api_source'])
    df.to_csv('datos/cache_autores.csv', index=False, encoding="utf-8")
    print(f" Cache de autores guardada en datos/cache_autores.csv. Total autores: ({len(registros)} en registros)")

#cargar cache al iniciar (para no repetir llamadas a la api)
cargar_cache_autores()

# 3 Creacion de la  base de datos (DDL "Definicion de Datos Lenguaje")
- La funcion crear base datos entra en accion.
- Construye las tablas mientras hablas:
- CategorÃ­as, libros y autores crear sin vacilar.
- Con autor_libro logra conectar
- Sus claves primarias y foreaneas al relacionar.
- Genera como vez sus i­ndices con gran agilidad
- Para buscar todo con mas velocidad encontrando su identidad.
- Con un commit guarda cada elemento,
- cierra la conexion y confirma el argumento del documento al momento.

In [12]:
def crear_base_datos():
#3.1 abre la base de datos, preparan un cursor para ejecutar SQL y activan la integridad referencial entre tablas.
    conexion=obtener_conexion()
    cursor=conexion.cursor()

#3.2.1 tabla categorias con sqlite3
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS categorias (
        id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_categoria TEXT UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

#3.2.2 tabla libros
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS libros(
            id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo_libro TEXT NOT NULL,
            precio_libro REAL NOT NULL,
            calificacion_libro INTEGER,
            categoria_id INTEGER,
            url_libro TEXT UNIQUE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (categoria_id) REFERENCES categorias(id_categoria)
        )
    ''')

#3.2.3 tabla autores (enriquesimiento)
    cursor.execute('''
        CREATE  TABLE IF NOT EXISTS autores(
            id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre_autor TEXT NOT NULL UNIQUE,
            pais_autor TEXT,
            api_external_id TEXT,
            api_source TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )    
    ''')

#3.2.4 tablas intermedias relacion muchos a muchos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS autor_libro (
            id_libro INTEGER,
            id_autor INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (id_libro, id_autor),
            FOREIGN KEY (id_libro) REFERENCES libros(id_libro),
            FOREIGN KEY (id_autor) REFERENCES autores(id_autor)
        )
    ''')

#3.3 indices para optimizar consultas
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_categoria ON libros(categoria_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_autores_nombre ON autores(nombre_autor)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_calificacion ON libros(calificacion_libro)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_autores_pais ON autores(pais_autor)')

#3.4 commit y cerrar conexion
    conexion.commit()
    conexion.close()
    print(" Base de datos y tablas creadas/verificadas.")

crear_base_datos()

 Base de datos y tablas creadas/verificadas.


# 4. Funciones de scraping ( categori­as, – libros y ¸ autores)
- extraer_categoria(): Busca en la pagina principal con alegri­a, las categori­as y sus rutas cada di­a.
- extraer_libros_categoria(): Recorre cada categori­a y seccion (incluyendo paginacion) y realiza la extraccion ti­tulo, URL, precio y calificacion.
- extraer_autor_desde_libro(): En la pigina del libro se detiene en un sector, y del texto saca el nombre del autor.

In [13]:
#4.1 Funcion de extraccion extrae todas las categorias y sus URLs
def extraer_categoria(url_principal: str) -> Dict[str,str]:
    soup= obtener_html(url_principal)
    if not soup:
        print(f"-> No se pudo obtener el contenido HTML de la pagina principal: {url_principal}")
        return{}

    categorias={}
    barra_lateral=soup.find('div',class_='side_categories')
    if barra_lateral:
        enlaces=barra_lateral.find_all('a')
        for enlace in enlaces:
            nombre=enlace.text.strip()
            if nombre.lower()!='books':
                url_relativa=enlace.get("href")
                url_completa=urljoin(url_principal,url_relativa)
                categorias[nombre]=url_completa

    print(f"Se encontro: {len(categorias)} Categorias")
    return categorias

#4.2 Extrae todos los libros de una categori­a (con paginacion).
def extraer_libros_categoria(url_categoria:str,categoria_nombre:str)->List[Dict]:
    libros=[]
    pagina_actual=url_categoria

    while pagina_actual:
        soup= obtener_html(pagina_actual)
        if not soup:
            print(f"-> No se pudo obtener el contenido HTML de la pagina principal: {pagina_actual}")
            break

        articulos=soup.find_all('article',class_='product_pod')
        for articulo in articulos:
            enlace_titulos= articulo.find('h3').find('a')
            if (not enlace_titulos):
                print(f"-> No se encontro el enlace del ti­tulo en el arti­culo: {articulo}")
                continue

            titulo=enlace_titulos.get('title')
            url_libro=urljoin(pagina_actual,enlace_titulos.get('href'))

            precio_tag=articulo.find('p', class_='price_color')
            precio_texto=precio_tag.text.strip() if precio_tag else '£0.00'
            precio=float(re.sub(r'[^\d.]', '', precio_texto))

            rating_tag=articulo.find('p',class_='star-rating')

            if rating_tag:
                clases= rating_tag.get('class')
                calificacion_texto= clases[1] if len(clases)>1 else 'Zero'
                mapeo={'Zero':0,'One':1,'Two':2,'Three':3,'Four':4,'Five':5,}
                calificacion=mapeo.get(calificacion_texto,0)
            else:
                calificacion=0

            libros.append({
                'titulo':titulo,
                'url':url_libro,
                'precio':precio,
                'calificacion':calificacion,
                'categoria':categoria_nombre
            })

        siguiente=soup.find('li',class_='next')
        if siguiente:
            enlace_siguiente=siguiente.find('a')
            if enlace_siguiente:
                pagina_actual=urljoin(pagina_actual,enlace_siguiente.get('href'))
                esperar(0.5)
            else:
                pagina_actual=None
        else:
            pagina_actual=None
    print(f" Categoria {categoria_nombre}: {len(libros)} Libros.")
    return libros

# 4.3 Extrae el nombre del autor desde la pÃ¡gina del libro.
def extraer_autor_desde_libro(url_libro: str)-> Optional[str]:
    soup= obtener_html(url_libro)
    if not soup:
        return None

    # Estrategia 1: Buscar <li class="author">
    autor_tag=soup.find('li', class_='author')# si o si en ingles para hacer referencia a la pagina web
    if autor_tag:
        enlace_autor=autor_tag.find('a')
        if enlace_autor:
            return enlace_autor.text.strip()
        # si no hay <a>,tomar texto directamente
        texto=autor_tag.get_text(strip=True)
        if texto and texto!='Author:':
            return texto

    #Estrategia 2 buscar <p class="author"
    autor_tag =soup.find('p',class_='authors')
    if autor_tag:
        enlace_autor=autor_tag.find('a')
        if enlace_autor:
            return enlace_autor.text.strip()
        texto=autor_tag.get_text(strip=True)
        if texto and texto!='Authors:':
            return texto

    #Estrategia 3 Buscar cualquier enlace que contenga "author" en la URL
    for enlace in soup.find_all('a', href=True):
        if 'author' in enlace['href'].lower():
            return enlace.text.strip()

    contenido=soup.get_text()
    if 'Author:' in contenido:
        coincidencia=re.search(r'Author:\s*([^\n]+)',contenido)
        if coincidencia:
            nombre_autor= coincidencia.group(1).strip()
            nombre_autor=nombre_autor.split('|')[0].split('•')[0].strip()
            return nombre_autor

    return None

## 5. Enriquecimiento de autores — SOLO Wikipedia

Como `books.toscrape.com` no trae el autor en el HTML, este es el flujo real para conseguirlo:

1. `es_nombre_valido()` — filtra nombres "basura" (series, texto con `#` o números) antes de usarlos para cualquier cosa.
2. `buscar_autor_por_titulo_wikipedia()` — busca el TÍTULO del libro en Wikipedia y trata de sacar el nombre del autor del resumen de la página (del libro).
3. `enriquecer_autor_wikipedia()` — con el nombre YA VALIDADO del autor, busca SU PROPIA página de Wikipedia y saca el país de nacimiento/nacionalidad. Guarda el título de esa página como `api_external_id` y `"wikipedia"` como `api_source`.
4. `enriquecer_autor()` — junta todo, usa la caché para no repetir llamadas, y nunca guarda ni consulta nombres inválidos.


In [14]:
# 5.1 Limpia el título del libro sacándole el sufijo "(Serie #N)" al final, que ensuciaría la búsqueda por título en Wikipedia.
def limpiar_titulo_para_busqueda(titulo:str)->str:
    return re.sub(r'\s*\([^)]*\)\s*$', '', titulo).strip()

#5.2 Valida si un texto parece realmente el nombre de una persona
def es_nombre_valido(nombre:str)->bool: 
    if not nombre:
        return False 

    nombre=nombre.strip()
    if not nombre or nombre=="Autor Desconocido":
        return False

    if len(nombre)>60 or any(c.isdigit() for c in nombre):
        return False

    if any(c in nombre for c in ['@','#','$','%','*','_','~','^', '(', ')', '/']):
        return False

    return bool(re.search(r"^[A-Za-zÀ-ÿ\.\-' ]+$", nombre))

#5.3 Wikipedia busca el título del libro y extrae el autor del resumen
def buscar_autor_por_titulo_wikipedia(titulo_libro: str)-> Optional[str]:
    titulo_busqueda = limpiar_titulo_para_busqueda(titulo_libro)
    parametros = {
        'action': 'query',
        'generator': 'search',
        'gsrsearch': titulo_busqueda,
        'gsrlimit': 5,
        'prop': 'extracts',
        'exintro': 1,
        'explaintext': 1,
        'format': 'json',
        'utf8': 1
    }

    try:
        respuesta=sesion.get('https://en.wikipedia.org/w/api.php', params=parametros, timeout=10)
        respuesta.raise_for_status()
        paginas=respuesta.json().get('query',{}).get('pages',{})
        if not paginas:
            return None

        patrones_autor = [
            r'is\s+an?\s+(?:novel|book|memoir|travelogue)\s+(?:by|written by)\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})',
            r'written\s+by\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})',
            r'\bby\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})[\.,]'
        ]
####aqui lo dejo
        for datos_pagina in paginas.values():
                extracto=datos_pagina.get('extract','')
                if not extracto:
                    continue
                for patron in patrones_autor:
                    coincidencia=re.search(patron, extracto)
                    if coincidencia:
                        nombre_candidato=coincidencia.group(1).strip()
                        if es_nombre_valido(nombre_candidato):
                            return nombre_candidato
        return None

    except requests.exceptions.HTTPError as e:
        print(f"Wikipedia (por título): error HTTP {e.response.status_code} para '{titulo_busqueda}'")
    except requests.exceptions.Timeout:
        print(f"Wikipedia (por título): tiempo de espera agotado para '{titulo_busqueda}'")
    except requests.exceptions.RequestException as e:
        print(f"Wikipedia (por título): error de red para '{titulo_busqueda}': {e}")
    return None

#5.4 Con el nombre validado del autor, busca su página de Wikipedia y extrae el país
def enriquecer_autor_wikipedia(nombre_autor: str) -> Optional[Dict]:
    parametros_busqueda = {
        'action': 'query',
        'generator': 'search',
        'gsrsearch': nombre_autor,
        'gsrlimit': 1,
        'prop': 'extracts',
        'exintro': 1,
        'explaintext': 1,
        'format': 'json',
        'utf8': 1
    }

    try:
        respuesta=sesion.get('https://en.wikipedia.org/w/api.php', params=parametros_busqueda, timeout=10)
        respuesta.raise_for_status()
        paginas=respuesta.json().get('query',{}).get('pages',{})
        if not paginas:
            return None

        datos_pagina=next(iter(paginas.values()))
        titulo_pagina=datos_pagina.get('title',nombre_autor)
        extracto=datos_pagina.get('extract','')
        
        patrones_pais=[
            r'born\s+in\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)',
            r'nationality\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)',
            r'is\s+an?\s+([A-Z][a-z]+)\s+(?:author|writer|novelist|poet)',
            r'from\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)'
        ]
        for patron in patrones_pais:
            coincidencia=re.search(patron, extracto,re.IGNORECASE)
            if coincidencia:
                pais=coincidencia.group(1).split(',')[0].split('.')[0].strip()
                return {'pais':pais, 'api_id':titulo_pagina, 'api_source':'wikipedia'}

        return {'pais':None, 'api_id':titulo_pagina, 'api_source':'wikipedia'}

    except requests.exceptions.HTTPError as e:
        print(f"Wikipedia (autor): error HTTP {e.response.status_code} para '{nombre_autor}'")
    except requests.exceptions.Timeout:
        print(f"Wikipedia (autor): tiempo de espera agotado para '{nombre_autor}'")
    except requests.exceptions.RequestException as e:
        print(f"Wikipedia (autor): error de red para '{nombre_autor}': {e}")
    return None

#5.5 funcion principal de enriquecimiento
def enriquecer_autor(nombre_autor: str) -> Optional[Dict]:
    if not es_nombre_valido(nombre_autor):
        print(f"Autor '{nombre_autor}' no es un nombre válido. Se descarta.")
        return None

    if nombre_autor in cache_autores:
        return cache_autores[nombre_autor]

    resultado = enriquecer_autor_wikipedia(nombre_autor)
    cache_autores[nombre_autor] = resultado
    return resultado

 # 6 Persistencia en BD y guardado de respaldos CSV
maneja la base de datos de libros, autores y categorÃ­as: busca si existen y, si no ahi, va a crear; luego guarda el libro evitando duplicados por URL y relacionÃ¡ndolo con su autor.
 1. pide ID de esta categorÃ­a, si no existe, crÃ©ala y dame su nuevo ID)
 2. ID del autor; si no existe, crÃ©a con los datos que tengas y dame su nuevo ID
 3. Guarda el libro en BD evitando duplicados por URL si el libro existe (usando la URL como clave); asegura que la relaciÃ³n con el autor estÃ© en la tabla intermedia autor y Libro. Si es nuevo, lo inserta y crea la relaciÃ³n autor-libro

In [15]:
def obtener_id_categoria(conn, nombre_categoria:str)->int:
    cursor=conn.cursor()
    cursor.execute("SELECT id_categoria FROM categorias WHERE nombre_categoria = ?",(nombre_categoria,))
    fila=cursor.fetchone()
    if fila:
        return fila[0]

    cursor.execute("INSERT INTO categorias (nombre_categoria) VALUES (?)",(nombre_categoria,))
    conn.commit()
    return cursor.lastrowid

def obtener_id_autor(conn,nombre_autor:str, datos_autor:Optional[Dict])-> int:
    cursor=conn.cursor()
    cursor.execute("SELECT id_autor FROM autores WHERE nombre_autor=?",(nombre_autor,))
    fila=cursor.fetchone()
    if fila:
        id_autor = fila[0]
        # Actualizar país y datos de API si tenemos nueva información
        if datos_autor:
            pais = datos_autor.get('pais')
            api_id = datos_autor.get('api_id')
            api_source = datos_autor.get('api_source')
            if pais or api_id or api_source:
                cursor.execute('''
                    UPDATE autores SET 
                        pais_autor = COALESCE(?, pais_autor),
                        api_external_id = COALESCE(?, api_external_id),
                        api_source = COALESCE(?, api_source)
                    WHERE id_autor = ?
                ''', (pais, api_id, api_source, id_autor))
                conn.commit()
        return id_autor

    pais=None
    api_id=None
    api_source=None

    if datos_autor:
        pais=datos_autor.get('pais')
        api_id=datos_autor.get('api_id')
        api_source=datos_autor.get('api_source')
    
    cursor.execute('''
            INSERT INTO autores (nombre_autor,pais_autor,api_external_id,api_source)
            VALUES(?, ?, ?, ?)
        ''',(nombre_autor,pais,api_id,api_source))
    conn.commit()
    return cursor.lastrowid

#Evita duplicado por URL
def guardar_libro(conn,libro:Dict, id_categoria:int, id_autor:int):
    cursor=conn.cursor()
    cursor.execute("SELECT id_libro FROM libros WHERE url_libro=?",(libro['url'],))
    fila=cursor.fetchone()

    if fila:
        id_libro=fila[0]
        cursor.execute(
            "INSERT OR IGNORE INTO autor_libro(id_libro,id_autor) VALUES (?,?)",
            (id_libro,id_autor)
        )
        conn.commit()
        return

    cursor.execute('''
        INSERT INTO libros (titulo_libro,precio_libro,calificacion_libro,categoria_id,url_libro)
        VALUES (?,?,?,?,?)
    ''',(libro['titulo'],libro['precio'],libro['calificacion'],id_categoria,libro['url']))
    conn.commit()

    id_libro=cursor.lastrowid
    cursor.execute("INSERT INTO autor_libro (id_libro, id_autor) VALUES (?,?)", (id_libro,id_autor))
    conn.commit()

# 7 Proceso principal de scraping con checkpoint en CSV
Para no perder nada si hay un error, guardamos los libros en un servidor.
Al CSV va la informaciÃ³n cargada, y asÃ­ la red no vuelve a ser scrapeada.

- cargar_progreso() / guardar_progreso()
Guarda y recupera el estado del scrapeado en un JSON para poder retomarlo si se interrumpe.
*"En un JSON el estado va guardando, por si la red se corta ejecutando. Recuperas la ruta sin demora, y retomas la carga desde ahora."*

- scrapear_todo()
Ejecuta el scraping completo: obtiene categorÃ­as, libros, autores, enriquece con API y guarda todo en la base de datos. Soporta continuar desde donde quedÃ³ si continuar=True.

- exportar_libros_csv()
Toma la base completa sin reparo, y la pasa a un CSV limpio y claro. Con punto y coma los datos ordenados, Â¡y quedan los libros exportados!

In [16]:
ARCHIVO_PROGRESO='datos/scraping_progreso.json'
ARCHIVO_LIBROS_CSV='datos/libros_extraidos.csv'

#7.1 Carga el estado del scraping desde un archivo JSON.
def cargar_progreso():
    if os.path.exists(ARCHIVO_PROGRESO):
        with open(ARCHIVO_PROGRESO,'r')as f:
            return json.load(f)
    return{'categorias_procesadas':[],'libros_guardados':0}

def guardar_progreso(progreso):
    with open(ARCHIVO_PROGRESO,'w') as f:
        json.dump(progreso,f)

#7.2 guardado del progreso


# Ejecuta el scraping completo. Si continuar=True, retoma desde donde quedó.
def scrapear_todo(continuar:bool=True):
    url_principal="https://books.toscrape.com/index.html"
    progreso = cargar_progreso() if continuar else {'categorias_procesadas':[],'libros_guardados':0}

    conexion=obtener_conexion()

# Obtener Categorias
    categorias=extraer_categoria(url_principal)
# Lista de todas las categorias
    lista_categorias=list(categorias.items())

# Si ya hay categorías procesadas, las saltamos
    if continuar:
        procesadas=set(progreso.get('categorias_procesadas',[]))
        lista_categorias = [(nom, url) for nom,url in lista_categorias if nom not in procesadas]

    total_libros_guardados= progreso.get('libros_guardados',0)

    for nombre_categoria,url_categoria in lista_categorias:
        print(f"\n Procesando categoria: {nombre_categoria}")
        id_catategoria = obtener_id_categoria(conexion,nombre_categoria)
        libros = extraer_libros_categoria(url_categoria,nombre_categoria)

        for libro in libros:
            esperar(0.1)
            #extraer el autor desde el nombre del libro desde la pagina del libro
            autor_nombre= buscar_autor_por_titulo_wikipedia(libro['titulo'])
            #si no se encuentra intenta extrarer el titulo
            if autor_nombre: #a la API (si autor_nombre sigue siendo None, no hace falta esperar nada)
                esperar(0.2)

            # 3. Si seguimos sin nada válido, queda como desconocido
            #    (y NO llamamos a enriquecer_autor con basura)
            if es_nombre_valido(autor_nombre):
                datos_autor=enriquecer_autor(autor_nombre)
            else:
                autor_nombre="Autor Desconocido"
                datos_autor=None
            
            id_autor=obtener_id_autor(conexion,autor_nombre,datos_autor)
            guardar_libro(conexion,libro,id_catategoria,id_autor)

            total_libros_guardados+=1
            pais_mostrado=datos_autor.get('pais') if datos_autor else 'N/A'
            print(f"    ✅ {libro['titulo'][:40]}... - Autor: {autor_nombre} (País: {pais_mostrado})")
            
        # Marcar categoria como procesada
        progreso['categorias_procesadas'].append(nombre_categoria)
        progreso['libros_guardados']=total_libros_guardados
        guardar_progreso(progreso)
        guardar_cache_autores()

    conexion.close()
    print(f"\n Scraping completado. Total libros guardados: {total_libros_guardados}")
    
    exportar_libros_csv()

    # Guardar libros extraídos en CSV para respaldo
def exportar_libros_csv():
    conexion=obtener_conexion()
    consulta="""
        SELECT l.id_libro, l.titulo_libro, l.precio_libro, l.calificacion_libro,
                c.nombre_categoria as categoria,
                GROUP_CONCAT(a.nombre_autor, '; ') as autores
        FROM libros l
        JOIN categorias c ON l.categoria_id = c.id_categoria
        JOIN autor_libro al ON l.id_libro = al.id_libro
        JOIN autores a ON al.id_autor = a.id_autor
        GROUP BY l.id_libro
    """
    df=pd.read_sql_query(consulta,conexion)
    conexion.close()
    df.to_csv(ARCHIVO_LIBROS_CSV,index=False, encoding="utf-8")
    print(f"Libros exportados a {ARCHIVO_LIBROS_CSV} ({len(df)} registros).")

 # 8 Ejecutar el scraping
 - Si es la primera vez, ejecuta con continuar=False para empezar desde cero.
 - Si quieres retomar despuÃ©s de una interrupciÃ³n, usa continuar=True (por defecto)

In [17]:
scrapear_todo(continuar=False)
#scrapear_todo(continuar=True)

Contenido HTML obtenido correctamente de https://books.toscrape.com/index.html
Se encontro: 50 Categorias

 Procesando categoria: Travel
Contenido HTML obtenido correctamente de https://books.toscrape.com/catalogue/category/books/travel_2/index.html
 Categoria Travel: 11 Libros.
 Esperando 0.1 segundos...
    ✅ It's Only the Himalayas... - Autor: Autor Desconocido (País: N/A)
 Esperando 0.1 segundos...
    ✅ Full Moon over Noah’s Ark: An Odyssey to... - Autor: Autor Desconocido (País: N/A)
 Esperando 0.1 segundos...
 Esperando 0.2 segundos...
    ✅ See America: A Celebration of Our Nation... - Autor: John Quincy Adams Ward (País: None)
 Esperando 0.1 segundos...
 Esperando 0.2 segundos...
    ✅ Vagabonding: An Uncommon Guide to the Ar... - Autor: White Wolf Publishing (País: CCP by)
 Esperando 0.1 segundos...
 Esperando 0.2 segundos...
    ✅ Under the Tuscan Sun... - Autor: Random House (País: None)
 Esperando 0.1 segundos...
 Esperando 0.2 segundos...
    ✅ A Summer In Europe... - Aut

# 9 Consultas
- Libros baratos, buena calificaciÃ³n y menor a 10 â‚¬: *"Baratos, de diez euros y con gran nota, Â¡lectura buena que el bolsillo no agota!"*
- Autor peor calificado con peor promedio mÃ­nimo: *"El promedio mÃ¡s bajo viene a buscar, al autor que a nadie logrÃ³ conquistar."*
- CategorÃ­a con mayor precio promedio: *" Esta categorÃ­a se cotiza muy alto, Â¡ver su costo medio te hace dar un salto!"*
- Top 5 autores con mÃ¡s libros: *"Cinco autores prolÃ­ficos vas a encontrar, con tantos libros que no podrÃ¡s parar"*.
- PaÃ­s que produce mÃ¡s libros con rating: *" En crear buenas obras este paÃ­s es el rey, Â¡las mejores estrellas se las lleva por ley!"*

In [ ]:
#consulta SQL utiles
def ejecutar_consulta(consulta, parametros=()):
    conexion=obtener_conexion()
    cursor=conexion.cursor()
    cursor.execute(consulta,parametros)
    resultados=cursor.fetchall()
    conexion.close()
    return resultados

# 9.1. Libros con >3 estrellas y <£10
print("1. Libros con >3 estrellas y <£10:")
for fila in ejecutar_consulta("""
    SELECT titulo_libro, precio_libro, calificacion_libro
    FROM libros
    WHERE calificacion_libro > 3 AND precio_libro < 10 
    ORDER BY precio_libro ASC
""")[:10]:
    print(f" -> {fila[0]} - £{fila[1]} ({fila[2]}*)")

# 9.2. Autor con peor promedio (mi­nimo 5 libros)
print ("\n 2. Autor con peor promedio de rating (mi­nimo 5 Libros): ")
resultado = ejecutar_consulta("""
    SELECT a.nombre_autor, AVG(l.calificacion_libro), COUNT(*) as total
    FROM autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    JOIN libros l ON al.id_libro=l.id_libro
    GROUP BY a.id_autor 
    HAVING COUNT(*)>=5
    ORDER BY AVG(l.calificacion_libro) ASC
    LIMIT 1 """)

if resultado:
    for fila in resultado:
        print(f"{fila[0]} - Promedio: {fila[1]:.2f}* ({fila[2]} libros)")
else:
    print(" Ningun autor tiene 5 o mas libros en esta categoria")

# 9.3. Categoria con mayor precio Promedio
print("\n 3. Categoria con mayor precio promedio: ")
for fila in ejecutar_consulta("""
    SELECT c.nombre_categoria, AVG(l.precio_libro)
    FROM categorias c
    JOIN libros l ON c.id_categoria=l.categoria_id
    GROUP BY c.id_categoria 
    ORDER BY AVG(l.precio_libro) DESC
    LIMIT 1
    """):
        print(f"    {fila[0]} - £{fila[1]:.2f} promedio")

# 9.4. los 5 autores con mas libros
print("\n 4. Top 5 Autores con mas libros: ")
for i, fila in enumerate(ejecutar_consulta("""
    SELECT a.nombre_autor, COUNT(*) as total
    from autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    GROUP BY a.id_autor
    ORDER BY total DESC
    LIMIT 5
"""),1):
        print(f"{i}. {fila[0]} - {fila[1]} libros")

# 9.5. CONSULTA OBLIGATORIA pais que produce mas libros con ranting >3
print("\n5. Pais con mas Libros de rating > 3 estrellas")
resultado= ejecutar_consulta("""
    SELECT a.pais_autor, COUNT(*) as Cantidad
    FROM autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    JOIN libros l ON al.id_libro=l.id_libro
    WHERE a.pais_autor IS NOT NULL AND l.calificacion_libro>3
    GROUP BY a.pais_autor 
    ORDER BY cantidad 
    DESC LIMIT 1
""")
if resultado:
    print(f"{resultado[0][0]} - {resultado[0][1]} libros")
else:
    print(" -> No hay datos de pai­s en la BD (API no lo proporciona).")

# 9.6 Autores sin Pais encotrados (registro de los casos no encontrados)
print("\n 6. Autores sin Pais (casos registrados como Null): ")
resultado=ejecutar_consulta(""" 
    SELECT COUNT(*)
    FROM autores
    WHERE pais_autor IS NULL
    """)
print(f" -> {resultado[0][0]} autores sin pais un total de"
    f"-> {ejecutar_consulta ('SELECT COUNT(*) FROM autores')[0][0]} autores.")

 # 10 Prueba de indexacion (antes/despues)

In [ ]:
# 10. Medicion de rendimiento con/sin i­ndices
import time

def medir_tiempo(consulta, descripcion):
    inicio=time.time()
    conexion=obtener_conexion()
    cursor=conexion.cursor()
    cursor.execute(consulta)
    cursor.fetchall()
    conexion.close()
    fin=time.time()
    print (f"{descripcion}: {fin - inicio:.4f} segundos")

# consulta Lenta busqueda por año de nacimiento (sin indice aun)

consulta_lenta="SELECT * FROM libros WHERE precio_libro BETWEEN 20 AND 30"

print("\n Antes de crear el Indice: ")
medir_tiempo(consulta_lenta,"Tiempo Sin Indice")

conexion=obtener_conexion()
plan_antes=conexion.execute(f"EXPLAIN QUERY PLAN {consulta_lenta}").fetchall()
conexion.execute("CREATE INDEX IF NOT EXISTS idx_libros_precio ON libros(precio_libro)")

conexion.commit()

plan_despues = conexion.execute(f"EXPLAIN QUERY PLAN {consulta_lenta}").fetchall()
conexion.close()

print("\n Medicion Despues del i­ndice: ")
medir_tiempo(consulta_lenta," Tiempo con Indice") 

print("\n Explicacion: El i­ndice reduce el tiempo de busqueda al permitir acceso directo a las filas que cumplen el rango de años.")